# 02 — Population-aware off-target: the reference-bias case

**AlleleForge is a research tool. It is not a medical device and provides no medical advice.**
Off-target nominations are computational and must be experimentally validated.

A guide validated as specific against the reference genome can still cut a site that *only exists in
some genomes*. A minor allele can create a **de-novo PAM** or restore a seed match, producing a real
off-target that a reference-only scan never sees — and the danger is often **ancestry-stratified**.

The canonical cautionary tale is the BCL11A enhancer variant **`rs114518452`** (Cancellieri,
Pinello et al., *Nat Genet* 2023): an allele enriched in African-ancestry genomes creates an `NGG`
PAM that yields a high-CFD off-target. This notebook reproduces the *finding* on a synthetic locus —
the locus is synthetic, the lesson is not.

In [ ]:
import tempfile
from pathlib import Path

from alleleforge.data.gnomad import GnomadDB, PopulationFrequency
from alleleforge.genome.reference import ReferenceGenome
from alleleforge.offtarget.engine import search
from alleleforge.types.guide import PAM

# A 20-nt spacer with no internal NRG PAM (so it never self-matches), and a
# synthetic contig where the reference protospacer is followed by "CGT" — i.e.
# NO NGG PAM on the reference. A reference-only scan must therefore be blind.
SPACER = "GACCATGCAACCTTGAACGT"
PAD = "T" * 10
contig = PAD + SPACER + "CGT" + PAD

fasta = Path(tempfile.mkdtemp()) / "ref.fa"
fasta.write_text(f">chr2\n{contig}\n")
reference = ReferenceGenome(fasta, build="hg38")
NGG = PAM(pattern="NGG")
print("contig:", contig)

## 1. A reference-only scan is blind

With only the reference in hand, the protospacer is followed by `CGT` — no `NGG` — so there is no
off-target to find. This is exactly the false sense of security the published finding warns about.

In [ ]:
ref_only = search(SPACER, NGG, reference=reference)
print("reference-only off-target sites:", ref_only.n_sites)
assert ref_only.n_sites == 0

## 2. Population variation creates a de-novo off-target

Now give the engine a population allele: an `rs114518452`-like variant, common in African-ancestry
genomes and rare elsewhere, that flips the base after the protospacer to create a `de-novo` `NGG`
PAM. The population-aware engine nominates the site, attributes it to the **causal allele**, and
attaches the per-ancestry frequencies.

In [ ]:
rs114518452 = PopulationFrequency(
    chrom="chr2",
    pos=32,
    ref="T",
    alt="G",
    overall_af=0.03,
    populations={"afr": 0.105, "amr": 0.012, "eas": 0.0, "nfe": 0.001, "sas": 0.0},
)
report = search(SPACER, NGG, reference=reference, gnomad=GnomadDB([rs114518452]))
site = report.sites[0]
print("population-aware sites :", report.n_sites)
print("origin                 :", site.origin.value)
print("CFD-style score        :", round(site.score, 3))
print("causal allele          :", site.causal_allele)
print("carrier populations    :", sorted(site.populations))

## 3. The risk is ancestry-stratified

The same edit is not equally risky for everyone. The site's hazard is concentrated in
African-ancestry genomes and near-absent in others — and populations below the MAF threshold do not
carry it at all. Reporting this stratification, rather than a single genome-wide number, is the
population-aware safety contract.

In [ ]:
print("per-ancestry frequency at the off-target site:")
for anc, freq in sorted(site.ancestries.items(), key=lambda kv: -kv[1]):
    bar = "#" * int(freq * 200)
    print(f"  {anc:4s} {freq:7.4f}  {bar}")
print()
print("worst-affected ancestry :", report.worst_ancestry())
print(
    "report stratification   :",
    {a: round(v, 4) for a, v in report.ancestry_stratification().items()},
)
assert site.ancestries["afr"] == max(site.ancestries.values())

## 4. One number for triage, with an auditable companion

Two guides with the same worst-case site can still differ in how *many* off-targets
they carry. `specificity_score()` rolls every nominated site into one comparable
number (Hsu-style `1/(1 + Σ site scores)`; **1.0** = no off-targets), the single
figure to triage a panel on. Each site also records its **MIT score beside the
primary CFD**, so a nomination kept by the engine's *either* CFD-or-MIT threshold
stays auditable rather than mysteriously retained.

In [ ]:
# Aggregate triage number + the per-site audit companion.
print("aggregate specificity :", round(report.specificity_score(), 4))
print("worst single-site CFD :", round(report.worst_score(), 4))
print("primary score method  :", site.score_method.value)
print("site MIT companion    :", site.mit_score)
# A perfectly-matched de-novo protospacer: MIT is defined (ungapped 20-nt) and 1.0.
assert site.mit_score == 1.0
assert 0.0 < report.specificity_score() <= 1.0

A reference-only pipeline would have shipped this guide as "clean." Searching population variation
by default — and reporting off-target risk stratified by ancestry — is how AlleleForge avoids
encoding reference bias into a therapeutic design. Swap the synthetic contig for a real hg38
reference and a gnomAD database and the call shape is identical.